# Task 1: 질문 유형 분류기 (Question Classification) — 직접 구현 BERT

충남대학교 Campus ChatBot — 질문을 5개 카테고리로 분류

| Label | 카테고리 |
|-------|----------|
| 0 | 졸업요건 |
| 1 | 학교 공지사항 |
| 2 | 학사일정 |
| 3 | 식단 안내 |
| 4 | 통학/셔틀 버스 |

- 모델: **직접 구현한 BERT** (klue/bert-base 가중치 copy + 임베딩 일치 검증)
- 학습: PyTorch 학습 루프 (5-class 분류)
- 평가: Macro-F1 + Confusion Matrix(Plotly)
- 입력: `data/test_cls.json` → 출력: `outputs/cls_output.json`


## 1. 환경 설정

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

import os

# 프로젝트 루트(data/ 폴더가 있는 곳)로 이동 — 제출폴더/Colab/로컬 모두 대응
_candidates = [os.getcwd(), os.path.dirname(os.getcwd()),
               "/content/submission", "/content/drive/MyDrive/cnu_qa_system"]
for _root in _candidates:
    if os.path.isdir(os.path.join(_root, "data")):
        os.chdir(_root)
        break
print("작업 디렉터리:", os.getcwd())

In [ ]:
import json
import os
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. 설정

In [ ]:
# ── 모델 설정 ──
MODEL_NAME = "klue/bert-base"
NUM_LABELS = 5
MAX_LENGTH = 128

# ── 학습 설정 ──
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

# ── 경로 설정 ──
TRAIN_PATH = "data/train_augmented.json"
VALID_PATH = "data/valid.json"
TEST_PATH = "data/test_cls.json"
OUTPUT_DIR = "outputs"
MODEL_SAVE_DIR = "model/classifier"

LABEL_NAMES = ["졸업요건", "학교 공지사항", "학사일정", "식단 안내", "통학/셔틀 버스"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 3. 데이터 로드 & 분포 확인

In [ ]:
def load_json(path: str) -> list:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


train_data = load_json(TRAIN_PATH)
valid_data = load_json(VALID_PATH)

print(f"학습 데이터: {len(train_data)}건")
print(f"검증 데이터: {len(valid_data)}건")

# 라벨 분포 확인
for split_name, split_data in [("train", train_data), ("valid", valid_data)]:
    counts = Counter(d["label"] for d in split_data)
    print(f"\n라벨 분포 ({split_name}):")
    for label_id in sorted(counts.keys()):
        print(f"  {label_id} ({LABEL_NAMES[label_id]}): {counts[label_id]}건")

## 4. 토크나이저 & 데이터셋

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class QuestionDataset(Dataset):
    def __init__(self, data: list, tokenizer, max_length: int = MAX_LENGTH):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        item = self.data[idx]
        encoding = self.tokenizer(
            item["question"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        result = {k: v.squeeze(0) for k, v in encoding.items()}
        if "label" in item:
            result["labels"] = torch.tensor(item["label"], dtype=torch.long)
        return result


train_dataset = QuestionDataset(train_data, tokenizer)
valid_dataset = QuestionDataset(valid_data, tokenizer)

print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

## 5. 모델 로드 & 학습

In [ ]:
# ════════ 직접 구현 BERT (소스: src/model/bert_scratch.py 인라인) ════════
# 직접 구현 BERT: 아키텍처 구현 → HF 가중치 copy → 임베딩 검증


import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class BertConfig:
    """BERT 하이퍼파라미터. HF BertConfig의 필요한 필드만 추린 것."""

    vocab_size: int = 32000
    hidden_size: int = 768
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    intermediate_size: int = 3072
    max_position_embeddings: int = 512
    type_vocab_size: int = 2
    layer_norm_eps: float = 1e-12
    hidden_dropout_prob: float = 0.1
    attention_probs_dropout_prob: float = 0.1
    pad_token_id: int = 0


class BertEmbeddings(nn.Module):
    """토큰 + 위치 + 세그먼트 임베딩을 합치고 LayerNorm한다."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id
        )
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(
        self, input_ids: torch.Tensor, token_type_ids: torch.Tensor | None = None
    ) -> torch.Tensor:
        seq_len = input_ids.size(1)
        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)

        embeddings = (
            self.word_embeddings(input_ids)
            + self.position_embeddings(position_ids)
            + self.token_type_embeddings(token_type_ids)
        )
        embeddings = self.LayerNorm(embeddings)
        return self.dropout(embeddings)


class BertSelfAttention(nn.Module):
    """멀티헤드 스케일드 닷-프로덕트 셀프어텐션."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)
        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def _shape(self, x: torch.Tensor) -> torch.Tensor:
        """(B, S, H) → (B, num_heads, S, head_size)."""
        b, s, _ = x.size()
        x = x.view(b, s, self.num_attention_heads, self.attention_head_size)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        q = self._shape(self.query(hidden_states))
        k = self._shape(self.key(hidden_states))
        v = self._shape(self.value(hidden_states))

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.attention_head_size)
        scores = scores + attention_mask  # 확장 마스크: pad 위치에 -10000
        probs = self.dropout(F.softmax(scores, dim=-1))

        context = torch.matmul(probs, v)  # (B, heads, S, head_size)
        context = context.permute(0, 2, 1, 3).contiguous()
        b, s, _, _ = context.size()
        return context.view(b, s, self.all_head_size)


class BertSelfOutput(nn.Module):
    """어텐션 출력 dense + residual + LayerNorm."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dropout(self.dense(hidden_states))
        return self.LayerNorm(hidden_states + input_tensor)


class BertAttention(nn.Module):
    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.self = BertSelfAttention(config)
        self.output = BertSelfOutput(config)

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        self_out = self.self(hidden_states, attention_mask)
        return self.output(self_out, hidden_states)


class BertIntermediate(nn.Module):
    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        return F.gelu(self.dense(hidden_states))  # exact gelu (HF 기본)


class BertOutput(nn.Module):
    """FFN 출력 dense + residual + LayerNorm."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dropout(self.dense(hidden_states))
        return self.LayerNorm(hidden_states + input_tensor)


class BertLayer(nn.Module):
    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.attention = BertAttention(config)
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        attn = self.attention(hidden_states, attention_mask)
        inter = self.intermediate(attn)
        return self.output(inter, attn)


class BertEncoder(nn.Module):
    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.layer = nn.ModuleList([BertLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layer:
            hidden_states = layer(hidden_states, attention_mask)
        return hidden_states


class BertPooler(nn.Module):
    """[CLS] 토큰 표현을 dense + tanh로 풀링."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        cls_token = hidden_states[:, 0]
        return torch.tanh(self.dense(cls_token))


class BertModel(nn.Module):
    """직접 구현한 BERT 인코더. HF BertModel과 동일한 서브모듈 이름 구조."""

    def __init__(self, config: BertConfig) -> None:
        super().__init__()
        self.config = config
        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)
        self.pooler = BertPooler(config)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns: (last_hidden_state, pooled_output)."""
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        # (B, S) → (B, 1, 1, S) 확장 마스크. pad(0) 위치를 -10000으로.
        ext_mask = attention_mask[:, None, None, :].to(dtype=torch.float32)
        ext_mask = (1.0 - ext_mask) * -10000.0

        embedding_output = self.embeddings(input_ids, token_type_ids)
        sequence_output = self.encoder(embedding_output, ext_mask)
        pooled_output = self.pooler(sequence_output)
        return sequence_output, pooled_output


class BertForQuestionClassification(nn.Module):
    """직접 구현한 BERT + 분류 헤드 (Task 1: 질문 5분류).

    HF BertForSequenceClassification과 같은 이름(bert/classifier)을 써,
    백본 가중치는 copy하고 분류 헤드만 새로 학습한다.
    """

    def __init__(self, config: BertConfig, num_labels: int = 5) -> None:
        super().__init__()
        self.num_labels = num_labels
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
    ) -> dict:
        _, pooled = self.bert(input_ids, attention_mask, token_type_ids)
        logits = self.classifier(self.dropout(pooled))
        out = {"logits": logits}
        if labels is not None:
            out["loss"] = F.cross_entropy(logits, labels)
        return out


# ── HF 가중치 copy & 검증 ────────────────────────────────────────────────────


def config_from_hf(hf_name: str) -> BertConfig:
    """HF 체크포인트의 config를 읽어 직접 구현한 BertConfig로 옮긴다."""
    from transformers import AutoConfig

    c = AutoConfig.from_pretrained(hf_name)
    return BertConfig(
        vocab_size=c.vocab_size,
        hidden_size=c.hidden_size,
        num_hidden_layers=c.num_hidden_layers,
        num_attention_heads=c.num_attention_heads,
        intermediate_size=c.intermediate_size,
        max_position_embeddings=c.max_position_embeddings,
        type_vocab_size=c.type_vocab_size,
        layer_norm_eps=getattr(c, "layer_norm_eps", 1e-12),
        hidden_dropout_prob=c.hidden_dropout_prob,
        attention_probs_dropout_prob=c.attention_probs_dropout_prob,
        pad_token_id=getattr(c, "pad_token_id", 0),
    )


def load_hf_backbone(model: BertModel, hf_name: str) -> None:
    """HF BertModel 가중치를 직접 구현한 BertModel로 copy한다.

    서브모듈 이름이 동일하므로 state_dict를 그대로 적재한다.
    (position_ids 등 HF의 비학습 버퍼는 제외하고 strict=False로 흡수)
    """
    from transformers import AutoModel

    hf_model = AutoModel.from_pretrained(hf_name)
    hf_state = hf_model.state_dict()
    # HF가 등록하는 비파라미터 버퍼 제거
    hf_state = {
        k: v
        for k, v in hf_state.items()
        if not k.endswith("position_ids") and "embeddings.token_type_ids" not in k
    }
    missing, unexpected = model.load_state_dict(hf_state, strict=False)
    # pooler를 안 쓰는 일부 체크포인트 대비 — pooler 외 누락은 에러로 본다
    real_missing = [k for k in missing if not k.startswith("pooler.")]
    if real_missing:
        raise RuntimeError(f"가중치 copy 누락: {real_missing[:5]} ...")
    if unexpected:
        print(f"[bert_scratch] 무시한 HF 키: {unexpected[:3]} ...")


@torch.no_grad()
def verify_against_hf(hf_name: str, text: str = "충남대학교 졸업요건 알려줘") -> float:
    """직접 구현한 BERT와 HF BERT의 last_hidden_state 최대 절대오차를 반환한다.

    1e-4 미만이면 forward 구현이 정확하다고 본다.

    Args:
        hf_name: HF 체크포인트 이름
        text: 검증용 입력 문장

    Returns:
        두 모델 출력의 최대 절대오차(float)
    """
    from transformers import AutoModel, AutoTokenizer

    config = config_from_hf(hf_name)
    mine = BertModel(config).eval()
    load_hf_backbone(mine, hf_name)

    hf_model = AutoModel.from_pretrained(hf_name).eval()
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    enc = tokenizer(text, return_tensors="pt")

    my_seq, _ = mine(enc["input_ids"], enc["attention_mask"], enc.get("token_type_ids"))
    hf_out = hf_model(**enc).last_hidden_state

    max_diff = (my_seq - hf_out).abs().max().item()
    print(f"[verify] 최대 절대오차 = {max_diff:.2e}  ({'OK ✅' if max_diff < 1e-4 else 'FAIL ❌'})")
    return max_diff


# ── 직접 구현 BERT + HF 가중치 copy ──
config = config_from_hf(MODEL_NAME)
model = BertForQuestionClassification(config, num_labels=NUM_LABELS)
load_hf_backbone(model.bert, MODEL_NAME)   # 백본만 copy, 분류 헤드는 새로 학습
model.to(DEVICE)
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")

# ── 임베딩 검증: 직접 구현 BERT == HF BERT ──
_diff = verify_against_hf(MODEL_NAME)
assert _diff < 1e-4, f"임베딩 불일치(구현 오류): {_diff}"

# ── 학습 준비 ──
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps
)
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


@torch.no_grad()
def eval_f1(model, loader):
    """검증셋 macro-F1과 (예측, 정답) 리스트를 반환한다."""
    model.eval()
    preds, gts = [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")
        logits = model(**batch)["logits"]
        preds += logits.argmax(-1).cpu().tolist()
        gts += labels.cpu().tolist()
    return f1_score(gts, preds, average="macro"), preds, gts


# ── 학습 루프 (best F1 모델 저장) ──
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
best_path = os.path.join(MODEL_SAVE_DIR, "pytorch_model.bin")
best_f1 = -1.0
print("학습 시작...")
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = model(**batch)["loss"]
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
    f1, _, _ = eval_f1(model, valid_loader)
    print(f"  epoch {epoch+1}/{EPOCHS}  loss={running/len(train_loader):.4f}  val_f1_macro={f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_path)
print(f"학습 완료! best val_f1_macro={best_f1:.4f}")

# best 모델 복원
model.load_state_dict(torch.load(best_path, map_location=DEVICE))


## 6. 검증 평가 & 모델 저장

In [ ]:
# ── 검증 평가 ──
val_f1, pred_labels, true_labels = eval_f1(model, valid_loader)
true_labels = np.array(true_labels)
pred_labels = np.array(pred_labels)
print(f"Validation F1 (macro): {val_f1:.4f}")
print(f"Validation F1 (weighted): {f1_score(true_labels, pred_labels, average='weighted'):.4f}")
print("\n분류 리포트:")
print(classification_report(true_labels, pred_labels, target_names=LABEL_NAMES, digits=4))

# ── Confusion Matrix (Plotly) ──
try:
    import plotly.figure_factory as ff
    cm = confusion_matrix(true_labels, pred_labels)
    fig = ff.create_annotated_heatmap(
        cm[::-1], x=LABEL_NAMES, y=LABEL_NAMES[::-1], colorscale="Blues"
    )
    fig.update_layout(title="Confusion Matrix (Validation)", xaxis_title="Predicted", yaxis_title="True")
    fig.show()
except Exception as e:
    print(f"[plotly 생략] {e}")

# ── 예측 결과 CSV 저장 [text, reference_label, predicted_label, correct] ──
import pandas as pd
df = pd.DataFrame({
    "text": [d["question"] for d in valid_data],
    "reference_label": [LABEL_NAMES[i] for i in true_labels],
    "predicted_label": [LABEL_NAMES[i] for i in pred_labels],
    "correct": (true_labels == pred_labels),
})
csv_path = os.path.join(OUTPUT_DIR, "cls_valid_predictions.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n검증 예측 CSV 저장: {csv_path}")

# ── 모델 저장 (state_dict + tokenizer) ──
torch.save(model.state_dict(), os.path.join(MODEL_SAVE_DIR, "pytorch_model.bin"))
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"모델 저장 완료: {MODEL_SAVE_DIR}")


## 7. 테스트셋 추론 → cls_output.json 생성

학습이 끝난 뒤, 저장된 모델을 로드하여 테스트셋을 추론한다.
평가 시에는 이 셀부터 실행해도 된다.

In [ ]:
# 저장된 직접 구현 BERT 분류기 로드
cls_config = config_from_hf(MODEL_NAME)
cls_model = BertForQuestionClassification(cls_config, num_labels=NUM_LABELS)
cls_model.load_state_dict(
    torch.load(os.path.join(MODEL_SAVE_DIR, "pytorch_model.bin"), map_location=DEVICE)
)
cls_model.to(DEVICE)
cls_model.eval()
cls_tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
print("분류 모델 로드 완료")


In [ ]:
def classify_questions(
    questions: list[str], model, tokenizer, device, batch_size: int = 32
) -> list[int]:
    """질문 리스트를 분류하여 라벨을 반환한다."""
    model.eval()
    all_preds = []
    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i : i + batch_size]
        encoding = tokenizer(
            batch_questions,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoding)
            preds = torch.argmax(outputs["logits"], dim=-1)
            all_preds.extend(preds.cpu().tolist())
    return all_preds


# 테스트 데이터 로드 & 추론
test_data = load_json(TEST_PATH)
test_questions = [d["question"] for d in test_data]
print(f"테스트 데이터: {len(test_questions)}건")

pred_labels = classify_questions(test_questions, cls_model, cls_tokenizer, DEVICE)

# 결과 생성 & 저장 (조교 양식: question, label)
cls_output = []
for item, label in zip(test_data, pred_labels):
    question = item["question"]
    cls_output.append({"question": question, "label": label})
    print(f"  [{label}] ({LABEL_NAMES[label]:8s}) {question}")

output_path = os.path.join(OUTPUT_DIR, "cls_output.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cls_output, f, ensure_ascii=False, indent=2)

print(f"\n결과 저장 완료: {output_path}")

## 8. 결과 검증

In [ ]:
with open(output_path, encoding="utf-8") as f:
    result = json.load(f)

print(f"cls_output.json: {len(result)}건")
print(f"샘플:\n{json.dumps(result[:3], ensure_ascii=False, indent=2)}")

pred_dist = Counter(d["label"] for d in result)
print("\n예측 라벨 분포:")
for label_id in sorted(pred_dist.keys()):
    print(f"  {label_id} ({LABEL_NAMES[label_id]}): {pred_dist[label_id]}건")

print("\n모든 라벨이 0~4 범위인지:", all(0 <= d["label"] <= 4 for d in result))

## 완료

- `model/classifier/pytorch_model.bin` — 직접 구현 BERT 분류기 가중치
- `outputs/cls_output.json` — 테스트셋 분류 결과 (조교 양식)
- `outputs/cls_valid_predictions.csv` — 검증셋 예측 [text, reference, predicted, correct]
